[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fm2-colab-benchmark/notebooks/m2/m2_colab_benchmark.ipynb)

# Benchmark M2 — Group 10 (CPU-only)

Notebook độc lập với export/recovery campaign. Parser đã sửa `nocite`, không dùng bản lỗi `072ee384`.
Chạy Configuration → Setup → Plan → Smoke một cohort → đọc audit → Run range. Chỉ chạy phạm vi đã xác nhận khi bấm Run.
Input chỉ đọc; output là campaign benchmark riêng. Không có API keys, crawler, Google Drive download hay Hugging Face upload.
Kết quả là quan sát thực tế, không bảo đảm mọi LaTeX parse được. Partial run không đại diện 200 cohort.

In [ ]:
# Chỉ sửa cấu hình tại đây; được đọc lại ở mỗi lần Run.
STUDENT_ID = "24120232"
START_COHORT = 1
END_COHORT = 200
COHORTS_PER_BATCH = 50
AUTO_CONTINUE = True
# Parent nhìn thấy archives của cả bốn thành viên (có thể là shortcuts/Shared Drive).
INPUT_DRIVE_DIR = "/content/drive/MyDrive/M1-pipeline-optimization-full"
OUTPUT_DRIVE_DIR = f"/content/drive/MyDrive/ArxivGroup10/{STUDENT_ID}/m2-benchmark"
# Để trống cho campaign mới. Resume campaign cũ: điền đúng đường dẫn và giữ
# nguyên range, INPUT_DRIVE_DIR, OUTPUT_DRIVE_DIR của lần chạy cũ.
BENCHMARK_CAMPAIGN = ""
# Ví dụ phạm vi cá nhân: START_COHORT=101; END_COHORT=150.
# Mỗi cohort cần .tar.gz + .manifest.json + .tar.gz.sha256 cạnh nhau.


## Setup — mount Drive và tải đúng release

Chọn CPU trong Runtime settings. Ô này chỉ mount Drive, tải source code/dependencies vào `/content` và kiểm tra runtime.
Không chạy parser, migration hay ghi campaign. Folder chia sẻ: thêm shortcut vào MyDrive hoặc dùng đường dẫn `/content/drive/Shareddrives/...` nhìn thấy được sau mount.
Không cần mở hoặc đọc `.env`.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys
import uuid

drive.mount("/content/drive")
RELEASE_COMMIT = "b01fe0ab6b433f3c2fe4f497a7318c4c8e7cd61a"
PARSER_FINGERPRINT = "4e88c7fb0d3463c4c6815b5afff3f64066815068dd4c34db1e92528966d60b70"
if os.environ.get("COLAB_GPU") == "1" or os.environ.get("COLAB_TPU_ADDR"):
    raise RuntimeError("Chọn runtime CPU-only rồi chạy lại Setup")
CODE_DIR = Path("/content/group10-m2-benchmark-" + RELEASE_COMMIT[:12])
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(CODE_DIR)], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", RELEASE_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
if head != RELEASE_COMMIT or subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Checkout không đúng release hoặc có local edits; không tự reset")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE_DIR / "requirements-m2-benchmark.txt")], check=True)
SCRATCH = "/content/group10-m2-benchmark-scratch-" + RELEASE_COMMIT[:12]
LOCAL_CONFIG = Path("/content/group10-m2-benchmark-config.json")
print("Release:", RELEASE_COMMIT, "Python:", sys.version.split()[0])


In [ ]:
def current_config():
    # Đọc giá trị hiện tại: sửa Configuration xong không bị capture giá trị cũ.
    return dict(repo=str(CODE_DIR), release_commit=RELEASE_COMMIT,
                start_cohort=START_COHORT, end_cohort=END_COHORT,
                input_drive_dir=INPUT_DRIVE_DIR, output_drive_dir=OUTPUT_DRIVE_DIR,
                scratch=SCRATCH, cohorts_per_batch=COHORTS_PER_BATCH,
                auto_continue=AUTO_CONTINUE, benchmark_campaign=BENCHMARK_CAMPAIGN)

def benchmark_command(action):
    LOCAL_CONFIG.write_text(json.dumps(current_config(), ensure_ascii=False), encoding="utf8")
    return [sys.executable, "-X", "utf8", "-B", "-m", "scripts.m2_colab_benchmark",
            "--config", str(LOCAL_CONFIG), "--action", action]

def plan():
    return json.loads(subprocess.check_output(benchmark_command("plan"), cwd=CODE_DIR, text=True))

def run_benchmark(action):
    info = plan()
    print("Operator:", STUDENT_ID, "Scope:", START_COHORT, END_COHORT)
    print("Effective COHORTS_PER_BATCH:", COHORTS_PER_BATCH, "AUTO_CONTINUE:", AUTO_CONTINUE)
    print("Campaign:", info["campaign"])
    print("Batches:", [(g[0], g[-1]) for g in info["batches"]])
    missing = [r for r in info["input_inventory"] if not r["available"]]
    if missing:
        print("Input chưa đủ (Run sẽ dừng trước parser):", json.dumps(missing, ensure_ascii=False))
    print("Existing markers (chưa re-audit):", info["existing_completions"])
    if action == "run":
        first = Path(info["campaign"]) / "completed" / (info["cohorts"][0] + ".json")
        if not first.is_file():
            raise RuntimeError("Chạy Smoke một cohort và kiểm tra audit trước Run range")
    result = subprocess.run(benchmark_command(action), cwd=CODE_DIR)
    if result.returncode:
        raise RuntimeError("Benchmark dừng an toàn; xem diagnostic/LATEST.json, không retry vô hạn")


## Plan — chỉ đọc

Kiểm tra đủ triplet của **toàn bộ phạm vi** trong parent dùng chung, không chọn tùy tiện khi có hai bản cùng cohort.
Plan chỉ kiểm tra sự hiện diện/đường dẫn; checksum nội dung được kiểm tra khi dùng input.
Completion markers chưa được coi là verified trước re-audit. Không đổi range/input path của campaign đang Resume.


In [ ]:
print(json.dumps(plan(), ensure_ascii=False, indent=2))

## Smoke — chỉ một cohort, ghi output benchmark

Ô này đọc/kiểm checksum archive, extract vào scratch local, parse mọi version cùng publication, ghi Parquet/checkpoints/reports mới trên Drive.
Không chạm campaign export cũ. Lỗi parse cô lập được giữ trong CSV; audit failure dừng cohort và giữ diagnostic.
Đọc `benchmark_summary.json`, `paper_results.csv`, `m2_errors.csv` và `audit_outcome` trước khi chạy tiếp.

In [ ]:
run_benchmark("smoke")

## Run batches / Resume — ghi output benchmark

Sau khi xem Smoke, chạy ô này một lần. `AUTO_CONTINUE=True` tự xử lý tuần tự từng nhóm
`COHORTS_PER_BATCH`, xuất report có `READY.json`, in đường dẫn rồi chạy nhóm tiếp theo.
`False` xử lý nhóm chưa hoàn tất kế tiếp rồi dừng. Chạy lại ô để Resume; cohort verified không parse lại.
Không đổi parser giữa các batch. Input/audit/timeout lỗi sẽ dừng; parse_error riêng từng version được ghi đúng thực tế.


In [ ]:
run_benchmark("run")

## Xuất CSV/JSON — không gọi parser

Dùng khi cần report partial hoặc chạy lại sau lỗi vẽ report ở release cũ. Chỉ đọc native evidence,
kiểm tra checksum rồi ghi snapshot mới khi dữ liệu thay đổi. Không ghi lại Parquet, không đọc archive để parse.
Không cần Matplotlib. Snapshot hoàn chỉnh được giữ nguyên; `LATEST.json` chỉ là con trỏ.


In [ ]:
run_benchmark("report")
info = plan()
latest = Path(info["campaign"]) / "LATEST.json"
if latest.is_file():
    print(latest.read_text(encoding="utf8"))

## Chỉ khi runtime cũ bị kill: kiểm tra locks (chỉ đọc)

Không chạy đồng thời hai runtime cùng campaign. Nếu còn lock, phải xác nhận process cũ đã dừng trước khi archive lock.
Hàm archive dưới đây chỉ chạy khi bạn gọi thủ công và nhập đúng run ID; không tự động gọi khi Run All.
Không đụng input, export/recovery hoặc checkpoint payload.

In [ ]:
def inspect_locks():
    root = Path(plan()["campaign"])
    if not root.exists():
        return []
    locks = sorted(p for p in root.rglob("*") if p.name in {".writer.lock", ".export-writer.lock"})
    for p in locks:
        if p.is_symlink() or not p.resolve().is_relative_to(root.resolve()):
            raise RuntimeError("Lock path không an toàn; dừng review")
        print(p, p.read_text(encoding="utf8", errors="replace"))
    return locks

def archive_stale_locks():
    root = Path(plan()["campaign"])
    locks = inspect_locks()
    if not locks:
        print("Không có lock")
        return
    answer = input("CHỈ khi runtime/process cũ đã dừng: nhập run ID để archive locks: ")
    if answer != root.name:
        raise RuntimeError("Không xác nhận; giữ nguyên locks")
    for p in locks:
        p.rename(p.with_name(p.name + ".reviewed-" + uuid.uuid4().hex))
        print("Đã archive lock evidence:", p)

inspect_locks()
# Nếu và chỉ nếu runtime cũ đã dừng: tự gọi archive_stale_locks() trong ô riêng.


## Đọc report và bàn giao Poster

Mỗi `reports/batch_XXXX-YYYY/snapshot-*/` có năm CSV/JSON dữ liệu, `report_checksums.json` và
`READY.json`. Chỉ lấy snapshot có READY và checksum hợp lệ. Có thể sao chép snapshot batch trước
cho Codex khi Colab đang chạy batch sau; không sửa snapshot hoặc parser đang chạy.
`reports/aggregate_XXXX-YYYY/` tính lại tổng từ cohort/version không trùng, không cộng phần trăm.
Không có PNG hay Markdown report. JSON ghi rõ mẫu số, status complete/partial và provenance từng SHA.

**Resume campaign cũ `benchmark-e7d87a5409189af18313`:** đặt lại đúng range 101–150,
INPUT_DRIVE_DIR cũ (thường là `.../M1-pipeline-optimization-full/24120232/archives`),
OUTPUT_DRIVE_DIR cũ và điền đầy đủ BENCHMARK_CAMPAIGN. Chạy lại Setup → Plan → Report → Run.
Helper chỉ tiếp nhận khi identity, parser fingerprint, provenance và evidence đều hợp lệ;
0101 được re-audit rồi skip, không gán SHA parser mới cho output cũ. Nếu scope/input khác: dừng, không xóa campaign.
Muốn 1–200 phải dùng campaign mới, không mở rộng âm thầm campaign 101–150.

Runtime mới: Configuration → Setup (hai ô) → Plan → Run; chỉ một writer. Nếu còn lock do hard kill,
xác minh runtime cũ đã dừng rồi dùng công cụ inspect/archive bên trên. Không tự xóa dữ liệu.
